# Medical Chatbot RAG — No OpenAI API

This notebook uses a local Hugging Face embedding model and Groq for the LLM.

In [1]:
# Cell 1 — Imports and API key
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
print("GROQ_API_KEY loaded:", bool(groq_api_key))

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found. Add GROQ_API_KEY=your_key to .env")

GROQ_API_KEY loaded: True


## 1. Load diabetes PDF

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "diabetes.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(
        f"{pdf_path} was not found. Put diabetes.pdf in the same folder as this notebook."
    )

loader = PyPDFLoader(pdf_path)
docs = loader.load()

print("PDF pages loaded:", len(docs))

C:\Users\ELCOT\AppData\Local\Temp\ipykernel_9672\2352268032.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF pages loaded: 23


## 2. Split the PDF into chunks

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(docs)
print("Chunks created:", len(chunks))

Chunks created: 36


## 3. Create local Hugging Face embeddings

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Hugging Face embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Hugging Face embedding model loaded successfully.


## 4. Create Chroma vector database

In [1]:
from langchain_chroma import Chroma

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding
)

print("Chroma vector database created successfully.")

NameError: name 'chunks' is not defined

## 5. Create retriever and test it

In [3]:
retriever = vectordb.as_retriever(
    search_kwargs={"k": 3}
)

query = "What is diabetes?"

retrieved_docs = retriever.invoke(query)

print("Retrieved documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content[:500])

NameError: name 'vectordb' is not defined

## 6. Create Groq LLM

In [6]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=groq_api_key
)

print("Groq LLM loaded successfully.")

NameError: name 'groq_api_key' is not defined

## 7. Create the medical RAG prompt

In [7]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a medical assistant. "
        "Answer using only the retrieved context below. "
        "If the answer is not available in the context, say you don't know. "
        "Do not invent medical facts. "
        "This chatbot provides educational information and is not a substitute "
        "for professional medical advice.\n\n"
        "Context:\n{context}"
    ),
    ("human", "{input}")
])

## 8. Build the RAG chain

In [8]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

combine_docs_chain = create_stuff_documents_chain(
    llm,
    prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    combine_docs_chain
)

print("Medical RAG chain created successfully.")

NameError: name 'llm' is not defined

## 9. Test the chatbot

In [10]:
query = "What is diabetes?"

result = rag_chain.invoke({
    "input": query
})

print("Answer:")
print(result["answer"])

print("\nSources:")
for doc in result["context"]:
    print("-", doc.metadata.get("source"))

NameError: name 'rag_chain' is not defined

## 10. Interactive Medical Chatbot

In [ ]:
while True:
    query = input("\nInput query (type 'exit' to stop): ").strip()

    if query.lower() == "exit":
        print("Chatbot stopped.")
        break

    if not query:
        continue

    result = rag_chain.invoke({
        "input": query
    })

    print("\nAnswer:")
    print(result["answer"])

    print("\nSources:")
    for doc in result["context"]:
        print("-", doc.metadata.get("source"))

KeyboardInterrupt: Interrupted by user